# Step 2: Steering Vector Extraction

For each persona × trait combination, extract contrastive steering vectors from
the model's residual stream.

- **6 personas** (5 axis + base model)
- **4 traits** (honesty, sycophancy, verbosity, formality)
- **14 layers** (middle third of residual stream)

Total: 6 × 4 × 20 pairs × 2 prompts = 960 forward passes (with multi-layer
capture per pass, so actual forward passes = 6 × 4 × 20 × 2 = 960).

In [ ]:
import torch
from nnsight import LanguageModel

from persona_steering.config import GEMMA_2_9B, Trait
from persona_steering.data import load_all_prompt_pairs
from persona_steering.extraction import SteeringVectorExtractor
from persona_steering.personas import load_all_personas
from persona_steering.utils import ensure_output_dirs, get_device, save_pickle

ensure_output_dirs()
device = get_device()
config = GEMMA_2_9B

In [ ]:
# Load model
model = LanguageModel(config.hf_id, device_map="auto", torch_dtype=torch.float16)
tokenizer = model.tokenizer

# Load personas and prompts
personas = load_all_personas()
prompt_pairs = load_all_prompt_pairs()

print(f"Personas ({len(personas)}): {[p.name for p in personas]}")
print(f"Traits ({len(prompt_pairs)}): {[t.value for t in prompt_pairs.keys()]}")
print(f"Extraction layers: {config.extraction_layers}")

In [ ]:
# Extract all vectors
extractor = SteeringVectorExtractor(model, tokenizer, config)
traits = list(Trait)
layers = config.extraction_layers

all_vectors = extractor.extract_all(personas, traits, prompt_pairs, layers)

# Summary
for persona_slug, trait_dict in all_vectors.items():
    for trait, layer_dict in trait_dict.items():
        mags = [vec.magnitude for vec in layer_dict.values()]
        print(f"  {persona_slug}/{trait.value}: {len(mags)} layers, "
              f"mag range [{min(mags):.3f}, {max(mags):.3f}]")

In [ ]:
# Save all vectors
from persona_steering.config import VECTORS_DIR

save_pickle(all_vectors, VECTORS_DIR / "all_vectors.pkl")

# Also save individually
for persona_slug, trait_dict in all_vectors.items():
    for trait, layer_dict in trait_dict.items():
        for layer, vec in layer_dict.items():
            vec.save()

print(f"Vectors saved to {VECTORS_DIR}")